# Train TRRUST Classifiers

Train binary (Relationship vs None) and ternary (Activation vs Repression vs None)
classifiers on TRRUST transcription factor–target gene relationships.

This notebook is **model-agnostic**: point `EMBEDDINGS_PATH` at any h5ad file
produced by `src/scgpt/encode.py` or `src/geneformer/encode.py` — both store
gene symbols in `obs_names` and embeddings in `.X`.

Training outputs (`train_losses`, `test_losses`, `classification_report`,
`gene_predictions`) live on the returned `TrainingResult` object and are
easy to serialize from a downstream notebook.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from src.trrust import (
    BINARY_LABELS,
    TERNARY_LABELS,
    load_binary_trrust_data,
    load_gene_embeddings,
    load_ternary_trrust_data,
    prepare_train_test_split,
    train_classifier,
)

## Configuration

Change `EMBEDDINGS_PATH` to swap foundation models — the rest of the notebook
does not depend on which model produced the embeddings.

In [ ]:
EMBEDDINGS_PATH = Path("../data/embeddings/scgpt_human_cd20.h5ad")
TRRUST_PATH = Path("../data/trrust_rawdata.human.tsv")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 64
BINARY_LR = 1e-5
TERNARY_LR = 1e-5
BINARY_EPOCHS = 20
TERNARY_EPOCHS = 20

print(f"Device: {DEVICE}")
print(f"Embeddings: {EMBEDDINGS_PATH}")
print(f"TRRUST: {TRRUST_PATH}")

## Load embeddings and TRRUST data

In [ ]:
gene_embeddings = load_gene_embeddings(EMBEDDINGS_PATH)
embsize = next(iter(gene_embeddings.values())).shape[0]

binary_data = load_binary_trrust_data(TRRUST_PATH, gene_embeddings)
ternary_data = load_ternary_trrust_data(TRRUST_PATH, gene_embeddings)

print(f"Gene embeddings: {len(gene_embeddings)} genes, {embsize}d")
print(f"Binary samples: {len(binary_data.records)}")
print(f"Ternary samples: {len(ternary_data.records)}")

## Binary classifier

90/10 stratified train/test split, no class weights.

In [ ]:
binary_train_ds, binary_test_ds, binary_test_meta = prepare_train_test_split(
    binary_data, test_size=0.1, seed=42,
)

binary_result = train_classifier(
    binary_train_ds,
    binary_test_ds,
    binary_test_meta,
    embsize=embsize,
    label_map=BINARY_LABELS,
    lr=BINARY_LR,
    epochs=BINARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=False,
    device=DEVICE,
)

In [ ]:
epochs = range(1, BINARY_EPOCHS + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, binary_result.train_losses, label="Train")
plt.plot(epochs, binary_result.test_losses, label="Test")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Binary Classifier Loss, Learning Rate = {BINARY_LR}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

print(pd.DataFrame(binary_result.classification_report).T.round(3))
print()
print("Sample predictions:")
print(binary_result.gene_predictions.head(10))

## Ternary classifier

90/10 stratified train/test split with inverse-frequency class weights to
handle the Activation/Repression/None imbalance.

In [ ]:
ternary_train_ds, ternary_test_ds, ternary_test_meta = prepare_train_test_split(
    ternary_data, test_size=0.1, seed=42,
)

ternary_result = train_classifier(
    ternary_train_ds,
    ternary_test_ds,
    ternary_test_meta,
    embsize=embsize,
    label_map=TERNARY_LABELS,
    lr=TERNARY_LR,
    epochs=TERNARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=True,
    device=DEVICE,
)

In [ ]:
epochs = range(1, TERNARY_EPOCHS + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, ternary_result.train_losses, label="Train")
plt.plot(epochs, ternary_result.test_losses, label="Test")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Ternary Classifier Loss, Learning Rate = {TERNARY_LR}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(pd.DataFrame(ternary_result.classification_report).T.round(3))
print()
print("Sample predictions:")
print(ternary_result.gene_predictions.head(10))